# YOLOv8s STC-Net Colab Experiment

This notebook runs the `STC-Net` version for YOLOv8s:

- DySnakeConv as the lightweight spatial-preserving front block
- Late-stage TAL regression reweighting
- Training-only auxiliary head for small `drill_pipe`
- QFL for classification
- AdamW optimizer

For a quick short run, you can lower `EPOCHS` and move `TAL_REG_START_EPOCH` earlier.
For a full run, keep `TAL_REG_START_EPOCH=150`.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/tuanziya666/yolov8s_kuangjing.git"
REPO_DIR = Path("/content/yolov8s_kuangjing")

if REPO_DIR.exists():
    %cd /content/yolov8s_kuangjing
    !git fetch origin
    !git reset --hard origin/main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd /content/yolov8s_kuangjing

!python -m pip uninstall -y ultralytics >/dev/null 2>&1 || true
!python -m pip install -q --no-deps -e .

In [ ]:
from pathlib import Path
import requests
from PIL import Image

assets_dir = Path("/content/yolov8s_kuangjing/ultralytics/assets")
assets_dir.mkdir(parents=True, exist_ok=True)

def ensure_valid_image(path: Path, url: str) -> None:
    valid = False
    if path.exists():
        try:
            with Image.open(path) as im:
                im.verify()
            valid = True
        except Exception:
            path.unlink(missing_ok=True)

    if not valid:
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        path.write_bytes(response.content)
        with Image.open(path) as im:
            im.verify()

ensure_valid_image(assets_dir / "bus.jpg", "https://ultralytics.com/images/bus.jpg")
ensure_valid_image(assets_dir / "zidane.jpg", "https://ultralytics.com/images/zidane.jpg")
print("assets fixed")

In [ ]:
import os
from pathlib import Path

MODEL_CFG = "ultralytics/cfg/models/v8/yolov8s_stcnet.yaml"
DATA_CFG = "ultralytics/cfg/datasets/coco8.yaml"  # Replace with your dataset yaml
EPOCHS = 40
IMGSZ = 640
BATCH = 16
WORKERS = 2
DEVICE = 0
PROJECT = "runs/colab"
NAME = "yolov8s_stcnet_colab"
SEED = 42

QFL_BETA = 2.0
TAL_REG_ENABLE = True
TAL_REG_START_EPOCH = 20  # use 150 for full long training
TAL_REG_EMA_DECAY = 0.95
TAL_REG_GAIN = 1.0
TAL_REG_THRESHOLD = 0.25

AUX_HEAD_ENABLE = True
AUX_HEAD_GAIN = 0.25
AUX_SMALL_AREA_THR = 0.012521
AUX_TARGET_CLASS_IDS = "2"

for key in [
    "ULTRALYTICS_CLS_LOSS",
    "ULTRALYTICS_QFL_BETA",
    "ULTRALYTICS_IOU_LOSS",
    "ULTRALYTICS_INNER_IOU_RATIO",
    "ULTRALYTICS_WCIOU_AC_LAMBDA",
    "ULTRALYTICS_WCIOU_AC_GAMMA",
    "ULTRALYTICS_SA_BOX_ENABLE",
    "ULTRALYTICS_SA_SMALL_ALPHA",
    "ULTRALYTICS_SA_ELONG_BETA",
    "ULTRALYTICS_SA_CLASS_GAMMA",
    "ULTRALYTICS_SA_SMALL_AREA_THR",
    "ULTRALYTICS_SA_ELONG_RATIO_THR",
    "ULTRALYTICS_SA_TARGET_CLASS_IDS",
    "ULTRALYTICS_TAL_REG_ENABLE",
    "ULTRALYTICS_TAL_REG_START_EPOCH",
    "ULTRALYTICS_TAL_REG_EMA_DECAY",
    "ULTRALYTICS_TAL_REG_GAIN",
    "ULTRALYTICS_TAL_REG_THRESHOLD",
    "ULTRALYTICS_AUX_HEAD_ENABLE",
    "ULTRALYTICS_AUX_HEAD_GAIN",
    "ULTRALYTICS_AUX_SMALL_AREA_THR",
    "ULTRALYTICS_AUX_TARGET_CLASS_IDS",
]:
    os.environ.pop(key, None)

os.environ["ULTRALYTICS_CLS_LOSS"] = "qfl"
os.environ["ULTRALYTICS_QFL_BETA"] = str(QFL_BETA)

os.environ["ULTRALYTICS_TAL_REG_ENABLE"] = "true" if TAL_REG_ENABLE else "false"
os.environ["ULTRALYTICS_TAL_REG_START_EPOCH"] = str(TAL_REG_START_EPOCH)
os.environ["ULTRALYTICS_TAL_REG_EMA_DECAY"] = str(TAL_REG_EMA_DECAY)
os.environ["ULTRALYTICS_TAL_REG_GAIN"] = str(TAL_REG_GAIN)
os.environ["ULTRALYTICS_TAL_REG_THRESHOLD"] = str(TAL_REG_THRESHOLD)

os.environ["ULTRALYTICS_AUX_HEAD_ENABLE"] = "true" if AUX_HEAD_ENABLE else "false"
os.environ["ULTRALYTICS_AUX_HEAD_GAIN"] = str(AUX_HEAD_GAIN)
os.environ["ULTRALYTICS_AUX_SMALL_AREA_THR"] = str(AUX_SMALL_AREA_THR)
os.environ["ULTRALYTICS_AUX_TARGET_CLASS_IDS"] = AUX_TARGET_CLASS_IDS

assert Path(MODEL_CFG).exists(), f"Missing model config: {MODEL_CFG}"
assert Path(DATA_CFG).exists(), f"Missing data config: {DATA_CFG}"
assert Path("ultralytics/nn/modules/block.py").exists(), "Missing ultralytics/nn/modules/block.py"
assert Path("ultralytics/nn/modules/head.py").exists(), "Missing ultralytics/nn/modules/head.py"
assert Path("ultralytics/nn/tasks.py").exists(), "Missing ultralytics/nn/tasks.py"
assert Path("ultralytics/utils/loss.py").exists(), "Missing ultralytics/utils/loss.py"
assert Path("ultralytics/engine/trainer.py").exists(), "Missing ultralytics/engine/trainer.py"
assert Path("ultralytics/models/yolo/detect/train.py").exists(), "Missing ultralytics/models/yolo/detect/train.py"

print("MODEL_CFG:", MODEL_CFG)
print("DATA_CFG:", DATA_CFG)
print("TAL_REG_START_EPOCH:", TAL_REG_START_EPOCH)
print("AUX_TARGET_CLASS_IDS:", AUX_TARGET_CLASS_IDS)

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_CFG)
results = model.train(
    data=DATA_CFG,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=PROJECT,
    name=NAME,
    pretrained="yolov8s.pt",
    cache=False,
    amp=False,
    patience=50,
    seed=SEED,
    deterministic=True,
    optimizer="AdamW",
    lr0=0.001,
    warmup_epochs=5,
    warmup_bias_lr=0.0,
)
results

In [ ]:
from pathlib import Path
import pandas as pd

run_dir = Path(PROJECT) / NAME
print("Run directory:", run_dir)
print("results.csv exists:", (run_dir / "results.csv").exists())
print("best.pt exists:", (run_dir / "weights" / "best.pt").exists())
print("last.pt exists:", (run_dir / "weights" / "last.pt").exists())

if (run_dir / "results.csv").exists():
    df = pd.read_csv(run_dir / "results.csv")
    extra_cols = [c for c in df.columns if c.startswith("tal/") or c.startswith("aux/")]
    display(df[["epoch"] + extra_cols].tail() if extra_cols else df.tail())